# Stage 3: ChromaDB Vector Store Population (CRAG Pipeline)
This notebook loads the unified `processed_chunks.json` generated in Stage 2, performs strict schema/uniqueness pre-flight checks, generates embeddings using `SentenceTransformer`, and populates a persistent `ChromaDB` vector collection.

In [ ]:
# Install required dependencies
!pip install -q chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [ ]:
import os
import json
import math
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer

# Define paths for Google Colab environment
DATA_DIR = Path("./data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

JSON_INPUT_PATH = DATA_DIR / "processed_chunks.json"
VECTORSTORE_DIR = DATA_DIR / "chroma_db"
COLLECTION_NAME = "vlm_papers"
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
BATCH_SIZE = 256  # Batched for GPU/CPU memory efficiency

In [ ]:
 def run_population_pipeline():
    # ------------------------------------------------------------------
    # Step 1: Load Combined File & Sanity Checks
    # ------------------------------------------------------------------
    if not JSON_INPUT_PATH.exists():
        raise FileNotFoundError(
            f"Missing input file at {JSON_INPUT_PATH}. "
            "Please upload 'processed_chunks.json' to the data directory."
        )

    with open(JSON_INPUT_PATH, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    total_chunks = len(chunks)
    print(f"[+] Loaded {total_chunks} total chunks from '{JSON_INPUT_PATH}'.")

    # Schema & Content Validation
    for i, c in enumerate(chunks):
        if not c.get("id"):
            raise ValueError(f"Malformed entry at index {i}: Missing or empty 'id'")
        if not c.get("text") or not c["text"].strip():
            raise ValueError(f"Malformed entry at index {i} (ID: {c.get('id')}): Empty 'text'")
        if "metadata" not in c or not isinstance(c["metadata"], dict):
            raise ValueError(f"Malformed entry at index {i} (ID: {c.get('id')}): Invalid 'metadata'")

    # Pre-flight ID Uniqueness Check
    ids_list = [c["id"] for c in chunks]
    if len(ids_list) != len(set(ids_list)):
        duplicates = len(ids_list) - len(set(ids_list))
        raise ValueError(f"[!] ID Collision Error: Found {duplicates} duplicate IDs across corpus.")

    print(f"[+] Sanity Check Passed: All {total_chunks} chunks valid with unique IDs.")

    # ------------------------------------------------------------------
    # Step 2: Collection Setup (Persistent & Idempotent)
    # ------------------------------------------------------------------
    client = chromadb.PersistentClient(path=str(VECTORSTORE_DIR))

    # Delete existing collection if present to prevent stale entries
    existing_collections = [col.name for col in client.list_collections()]
    if COLLECTION_NAME in existing_collections:
        client.delete_collection(name=COLLECTION_NAME)
        print(f"[+] Deleted existing collection '{COLLECTION_NAME}'.")

    collection = client.create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"}
    )
    print(f"[+] Created fresh collection '{COLLECTION_NAME}'.")

    # ------------------------------------------------------------------
    # Step 3: Extract Parallel Lists
    # ------------------------------------------------------------------
    ids = ids_list
    documents = [c["text"] for c in chunks]
    metadatas = [c["metadata"] for c in chunks]

    # ------------------------------------------------------------------
    # Step 4 & 5: Batched Embedding & Writing to Chroma
    # ------------------------------------------------------------------
    print(f"[+] Initializing embedding model: '{EMBEDDING_MODEL_NAME}'...")
    embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

    num_batches = math.ceil(total_chunks / BATCH_SIZE)
    print(f"[+] Starting batched processing ({num_batches} batch(es) of max size {BATCH_SIZE})...")

    for b_idx in range(num_batches):
        start_i = b_idx * BATCH_SIZE
        end_i = min(start_i + BATCH_SIZE, total_chunks)

        batch_ids = ids[start_i:end_i]
        batch_docs = documents[start_i:end_i]
        batch_meta = metadatas[start_i:end_i]

        # Batch Embed with L2 Normalization (Cosine Similarity)
        batch_embeddings = embedding_model.encode(
            batch_docs,
            show_progress_bar=False,
            normalize_embeddings=True
        ).tolist()

        # Write Batch to Chroma
        collection.add(
            ids=batch_ids,
            embeddings=batch_embeddings,
            documents=batch_docs,
            metadatas=batch_meta
        )
        print(f"    - Processed batch {b_idx + 1}/{num_batches} (Chunks {start_i} to {end_i - 1})")

    # ------------------------------------------------------------------
    # Step 6: Verification
    # ------------------------------------------------------------------
    stored_count = collection.count()
    print("\n" + "=" * 50)
    print(f"[+] VERIFICATION SUCCESSFUL")
    print(f"    - JSON Chunks Count:  {total_chunks}")
    print(f"    - Chroma Store Count: {stored_count}")

    assert stored_count == total_chunks, "Mismatch between input chunks and stored vectors!"

    # Test Query
    test_query = "What is reinforcement learning with CLIP feedback?"
    query_vector = embedding_model.encode([test_query], normalize_embeddings=True).tolist()

    results = collection.query(
        query_embeddings=query_vector,
        n_results=2
    )

    print("\n[+] Verification Similarity Search:")
    print(f"    Query: '{test_query}'")
    for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
        print(f"\n    Match #{i + 1} [Paper: {meta.get('source_paper')}, Section: {meta.get('section_name')}]:")
        print(f"    '{doc[:180]}...'")
    print("=" * 50)

# Run pipeline
run_population_pipeline()

[+] Loaded 3365 total chunks from 'data/processed_chunks.json'.
[+] Sanity Check Passed: All 3365 chunks valid with unique IDs.
[+] Created fresh collection 'vlm_papers'.
[+] Initializing embedding model: 'BAAI/bge-small-en-v1.5'...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[+] Starting batched processing (14 batch(es) of max size 256)...
    - Processed batch 1/14 (Chunks 0 to 255)
    - Processed batch 2/14 (Chunks 256 to 511)
    - Processed batch 3/14 (Chunks 512 to 767)
    - Processed batch 4/14 (Chunks 768 to 1023)
    - Processed batch 5/14 (Chunks 1024 to 1279)
    - Processed batch 6/14 (Chunks 1280 to 1535)
    - Processed batch 7/14 (Chunks 1536 to 1791)
    - Processed batch 8/14 (Chunks 1792 to 2047)
    - Processed batch 9/14 (Chunks 2048 to 2303)
    - Processed batch 10/14 (Chunks 2304 to 2559)
    - Processed batch 11/14 (Chunks 2560 to 2815)
    - Processed batch 12/14 (Chunks 2816 to 3071)
    - Processed batch 13/14 (Chunks 3072 to 3327)
    - Processed batch 14/14 (Chunks 3328 to 3364)

[+] VERIFICATION SUCCESSFUL
    - JSON Chunks Count:  3365
    - Chroma Store Count: 3365

[+] Verification Similarity Search:
    Query: 'What is reinforcement learning with CLIP feedback?'

    Match #1 [Paper: 2305.18010v2, Section: 1 Introduction]